# Object-Oriented Python, Part 2 (8/18/26)

---

*Codecademy — MLE Path: SWE for ML → **Object-Oriented Programming**. Concise grad-student notes. Continues [OOP Part 1](OOP_Python_Lesson.ipynb) (Introduction to Classes).*

> **Scope:** the four pillars of OOP — inheritance, polymorphism, abstraction, encapsulation.

## Object-Oriented Programming

### Introduction to Object-Oriented Programming

A **programming paradigm** is a way of classifying languages by the features they offer. Most modern languages support several at once, so the same Python file can sit in more than one category.

**Object-Oriented Programming (OOP)** is the paradigm we've already been using: build programs around **classes** and **objects**, modeling real-world entities as classes with **properties** (data) and **methods** (behavior).

Recap of what Part 1 covered, in one class:

In [1]:
class Dog:
  sound = "Woof"

  def __init__(self, name, age):
    self.name = name
    self.age = age

  def bark(self):
    print(Dog.sound)

**Exercise** — an `Employee` class where every instance gets its own unique ID.

The trick is the **class variable as a counter**: `new_id` lives on the class (one copy, shared), so `__init__` can hand the current value to `self.id` and then bump it for the next instance.

In [2]:
# Write your code below
class Employee: 
  new_id = 1
  def __init__(self): 
    self.id =  Employee.new_id
    Employee.new_id += 1
  def say_id(self):
    print("My id is {}".format(self.id))

e1 = Employee()
e2 = Employee()
e1.say_id()
e2.say_id()
# -> My id is 1
# -> My id is 2

My id is 1
My id is 2


Here a real-world entity (a dog) is a class with properties (`name`, `age`) and a method (`bark`). `sound` is a class variable; `name` and `age` are instance variables set in `__init__`.

That's the core of OOP, but only the surface. The rest of the lesson works through the **four pillars**:

| Pillar | Idea |
| --- | --- |
| **Inheritance** | a class can take on another class's properties and methods |
| **Polymorphism** | the same interface behaves differently depending on the object |
| **Abstraction** | expose what a thing does, hide how it does it |
| **Encapsulation** | bundle data with the methods that touch it, and restrict outside access |

### Pillar 1: Inheritance

Two classes that share behavior mean duplicated code:

```
class Dog:
  def bark(self):
    print('Woof!')

class Cat:
  def meow(self):
    print('Meow!')
```

Give both an `eat()` method by writing it twice? That repeats code, and repeats again in every new animal class. **Inheritance** solves it: pull shared behavior up into a **parent class**, and let **child classes** receive it.

```
class ParentClass:
  # methods / properties...

class ChildClass(ParentClass):
  # methods / properties...
```

The parent goes in parentheses after the child's name.

In [3]:
class Animal:
  def eat(self):
    print("Nom Nom Nom...eating food!")

class Dog(Animal):
  def bark(self):
    print('Bark!')

class Cat(Animal):
  def meow(self):
    print('Meow!')

fluffy = Dog()
zoomie = Cat()

fluffy.eat()   # -> Nom Nom Nom...eating food!
zoomie.eat()   # -> Nom Nom Nom...eating food!

fluffy.bark()  # -> Bark!
zoomie.meow()  # -> Meow!

Nom Nom Nom...eating food!
Nom Nom Nom...eating food!
Bark!
Meow!


`Dog` and `Cat` never define `eat()`, but both have it — Python looks on the instance, doesn't find it, then looks on the parent.

Two payoffs: **reuse** methods across many classes from one definition, and **model parent-child relationships** between entities (a dog *is an* animal).

**Exercise** — make `Admin` a more specific kind of `Employee`.

`Admin` adds nothing of its own (`pass`), yet an `Admin` instance still gets `__init__` and `say_id()` from the parent. Note the counter is on `Employee`, so `Admin` shares the same ID sequence — `e3` is the third employee made.

In [4]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

# Write your code below
class Admin(Employee):
  pass


e1 = Employee()
e2 = Employee()
e3 = Admin()
e3.say_id()
# -> My id is 3.

My id is 3.


### Overriding Methods

A child class often wants different behavior than the parent. To **override** a method, just define it again in the subclass with the same name — same signature, different body. The child's version wins.

`Cat` inherits `__init__` (so `self.name` still works) but replaces `make_noise()`.

In [5]:
class Animal:
  def __init__(self, name):
    self.name = name

  def make_noise(self):
    print("{} says, Grrrr".format(self.name))

class Cat(Animal):

  def make_noise(self):
    print("{} says, Meow!".format(self.name))

pet1 = Animal("Rex")
pet1.make_noise()   # -> Rex says, Grrrr

pet2 = Cat("Maisy")
pet2.make_noise()   # -> Maisy says, Meow!

Rex says, Grrrr
Maisy says, Meow!


`Cat` has everything `Animal` has; the only difference is which `make_noise()` gets found first. Python checks the child class before the parent, so the subclass definition shadows the inherited one.

**Exercise** — `Admin` overrides `say_id()` to announce the role instead of the ID.

`e3` still gets an ID from the inherited `__init__` (it's the third employee), but `Admin.say_id()` shadows the parent's, so the ID never gets printed.

In [6]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

class Admin(Employee):
  # Write your code below
  def say_id(self):
    print("I am an Admin")
    
  

e1 = Employee()
e2 = Employee()
e3 = Admin()
e3.say_id()
# -> I am an Admin

I am an Admin


### `super()`

Overriding replaces the parent's method entirely. Often you want the parent's behavior *plus* something extra. `super()` gives you a proxy object standing in for the **superclass**, so you can call the parent version by name:

```
super().method_name(args)
```

In [7]:
class Animal:
  def __init__(self, name, sound="Grrrr"):
    self.name = name
    self.sound = sound

  def make_noise(self):
    print("{} says, {}".format(self.name, self.sound))

class Cat(Animal):
  def __init__(self, name):
    super().__init__(name, "Meow!") 

pet_cat = Cat("Rachel")
pet_cat.make_noise()   # -> Rachel says, Meow!

Rachel says, Meow!


What happens here:

- `Cat` defines its own `__init__`, which **overrides** `Animal.__init__` — so the parent's setup would not run at all.
- `super().__init__(name, "Meow!")` calls it anyway, passing the name through and hard-coding the sound.
- `self.name` and `self.sound` therefore still get set, and the inherited `make_noise()` works unchanged.

Use `super()` when the subclass needs the superclass's behavior **alongside** its own, not instead of it. It's most common in `__init__`, where the parent usually has setup the child still needs.

**Exercise** — `Admin` should say its ID *and* that it's an admin.

Same override as before, but now `super().say_id()` runs the parent's version first, then the subclass adds its own line. Parent behavior plus extra, rather than instead of.

In [8]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

class Admin(Employee):
  def say_id(self):
    # Write your code below:
    super().say_id()
    print("I am an admin.")

e1 = Employee()
e2 = Employee()
e3 = Admin()
e3.say_id()
# -> My id is 3.
# -> I am an admin.

My id is 3.
I am an admin.


### Multiple Inheritance, Part 1: inheritance chains

**Multiple inheritance** is a subclass inheriting from more than one superclass. One form of it is **multiple levels** of inheritance — a class inherits from its superclass *and* its super-superclass.

In [9]:
class Animal:
  def __init__(self, name):
    self.name = name
 
  def say_hi(self):
    print("{} says, Hi!".format(self.name))

class Cat(Animal):
  pass

class Angry_Cat(Cat):
  pass

my_pet = Angry_Cat("Mr. Cranky")
my_pet.say_hi()   # -> Mr. Cranky says, Hi!

Mr. Cranky says, Hi!


`Angry_Cat` → `Cat` → `Animal`. Both `Angry_Cat` and `Cat` get `name` and `say_hi()` from `Animal`, even though neither defines anything itself. Lookup walks up the chain until it finds the name.

Anything added to `Cat` later is automatically available to `Angry_Cat` too.

**Exercise** — a three-level chain: `Manager` → `Admin` → `Employee`.

Each level's `say_id()` calls `super().say_id()`, so one call on `e4` cascades all the way up and you get output from all three classes.

In [10]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

class Admin(Employee):
  def say_id(self):
    super().say_id()
    print("I am an admin.")

# Write your code below
class Manager(Admin):
  def say_id(self):
    print("They are in charge")
    super().say_id()


e1 = Employee()
e2 = Employee()
e3 = Admin()
e4 = Manager()
e4.say_id()
# -> They are in charge
# -> My id is 4.
# -> I am an admin.

They are in charge
My id is 4.
I am an admin.


Order follows where each `super()` call sits in the body. `Manager` prints first *then* delegates up, while `Admin` delegates up *then* prints — so `Employee`'s line lands in the middle.

### Multiple Inheritance, Part 2: two parents at once

The other form: a subclass inherits **directly from two classes** and can use the attributes and methods of both. List both parents in the parentheses.

In [11]:
class Animal:
  def __init__(self, name):
    self.name = name

class Dog(Animal):
  def action(self):
    print("{} wags tail. Awwww".format(self.name))

class Wolf(Animal):
  def action(self):
    print("{} bites. OUCH!".format(self.name))

class Hybrid(Dog, Wolf):
  def action(self):
    super().action()
    Wolf.action(self)

my_pet = Hybrid("Fluffy")
my_pet.action()   # -> Fluffy wags tail. Awwww
                  # -> Fluffy bites. OUCH!

Fluffy wags tail. Awwww
Fluffy bites. OUCH!


`Hybrid` subclasses both `Dog` and `Wolf`, which are themselves both subclasses of `Animal`. All three can use `Animal`'s features; `Hybrid` can use `Dog`'s and `Wolf`'s. But `Dog` and `Wolf` cannot use each other's — they're siblings, not ancestors.

Two things to be careful about when both parents define the same method name:

- **`super()` picks the first parent listed.** `super().action()` inside `Hybrid` runs `Dog.action()`, because `Dog` comes before `Wolf` in `class Hybrid(Dog, Wolf)`. Reorder the parents and the behavior changes.
- **Calling the other parent means calling it on the class, and passing `self` yourself.** `Wolf.action(self)` — no instance is bound in that form, so `self` has to go in explicitly. That's what lets `Wolf.action` see the `Hybrid` instance and print the right name.

Useful for adding functionality from a class that doesn't fit the existing hierarchy, but the ambiguity it introduces is real — keep these structures shallow.

**Exercise** — `Admin` also needs to be a `User` of the site.

Inherit from both, with `Employee` listed first so `super()` still hits the existing chain. Then set up the user data by calling `User.__init__` directly — same pattern as `Wolf.action(self)` in the Hybrid demo.

In [12]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

class User:
  def __init__(self, username, role="Customer"):
    self.username = username
    self.role = role

  def say_user_info(self):
    print("My username is {}".format(self.username))
    print("My role is {}".format(self.role))

# Write your code below
class Admin(Employee, User):
  def __init__(self):
    super().__init__()
    User.__init__(self, self.id, "Admin")
    

  def say_id(self):
    super().say_id()
    print("I am an admin.")

e1 = Employee()
e2 = Employee()
e3 = Admin()
e3.say_user_info()
# -> My username is 3
# -> My role is Admin

My username is 3
My role is Admin


`Employee` is listed first, so `super().__init__()` still runs `Employee.__init__` (assigns id 3, bumps the counter). The second parent isn't on that `super()` path, so `User.__init__(self, self.id, "Admin")` is called on the class with `self` passed explicitly — same move as `Wolf.action(self)`. Username becomes the employee id; role is `"Admin"`.

### OOP Pillar: Polymorphism

**Polymorphism** is the ability to apply an identical operation onto different types of objects. That's useful when the object's type may not be known at runtime. Overriding a parent method is already one form of it.

In [13]:
class Animal:
  def __init__(self, name):
    self.name = name

  def make_noise(self):
    print("{} says, Grrrr".format(self.name))

class Cat(Animal):

  def make_noise(self):
    print("{} says, Meow!".format(self.name))

class Robot:
  
  def make_noise(self):
    print("beep.boop...BEEEEP!!!")

an_animal = Animal("Bear")
my_pet = Cat("Maisy")
my_vacuum = Robot()
objects = [an_animal, my_pet, my_vacuum]
for o in objects:
  o.make_noise()
# -> Bear says, Grrrr
# -> Maisy says, Meow!
# -> beep.boop...BEEEEP!!!

Bear says, Grrrr
Maisy says, Meow!
beep.boop...BEEEEP!!!


Same method name, different behavior. `Cat` overrides `Animal.make_noise()` (related by inheritance); `Robot` isn't related to either — it just happens to define the same method. Python only cares that the object responds to the call.

The payoff is the loop: a mixed list, one `.make_noise()` call per object, no type checks and no `if isinstance`. You don't need to know which class the method belongs to.

**Exercise** — schedule a meeting with at least one `Employee`, one `Admin`, and one `Manager`.

Put one instance of each in a list, then loop and call `.say_id()` on every item. Same call, three different outputs — that's the polymorphism.

In [14]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

  def say_id(self):
    print("My id is {}.".format(self.id))

class Admin(Employee):
  def say_id(self):
    super().say_id()
    print("I am an admin.")

class Manager(Admin):
  def say_id(self):
    super().say_id()
    print("I am in charge!")

# Write your code below
meeting = [Employee(), Admin(), Manager()]
for a in meeting:
  a.say_id()
# -> My id is 1.
# -> My id is 2.
# -> I am an admin.
# -> My id is 3.
# -> I am an admin.
# -> I am in charge!

My id is 1.
My id is 2.
I am an admin.
My id is 3.
I am an admin.
I am in charge!


The loop never asks which class `a` is. `Employee.say_id()` prints just the id; `Admin` and `Manager` each layer extra lines on top via `super()`. Instantiating inside the list literal still bumps `new_id` left to right, so the three ids are 1, 2, and 3.

### Dunder Methods

`+` already does different things depending on the type — that's **operator overloading**, another form of polymorphism:

```
2 + 4                        # int  → 6
"Is this " + "addition?"     # str  → "Is this addition?"
[1, 2] + [3, 4]              # list → [1, 2, 3, 4]
```

Python classes get a set of special methods that hook this behavior. They're called **dunder methods** (double underscores around the name). We've already used `__init__()` and `__repr__()`.

`__repr__(self)` takes only `self` and must return a string — the class's string representation. `print()` on an instance shows it.

Defining a class's dunder methods is how you overload operators.

In [15]:
class Animal:
  def __init__(self, name):
    self.name = name

  def __repr__(self):
    return self.name

  def __add__(self, another_animal):
    return Animal(self.name + another_animal.name)

a1 = Animal("Horse")
a2 = Animal("Penguin")
a3 = a1 + a2
print(a1) # Prints "Horse"
print(a2) # Prints "Penguin"
print(a3) # Prints "HorsePenguin"

Horse
Penguin
HorsePenguin


`a3 = a1 + a2` calls `a1.__add__(a2)` — the left operand's method, with the right operand passed as the argument. The names get concatenated and a new `Animal` is returned. `__repr__` is why `print(a3)` shows `HorsePenguin` instead of `<Animal object at 0x...>`.

**Exercise** — a `Meeting` that already overloads `+` to append employees. Make `len()` work on it too.

Define `__len__` so it returns `len(self.attendees)`, then add `e1`, `e2`, and `e3` with `+` and print the meeting's length.

In [16]:
class Employee():
  new_id = 1
  def __init__(self):
    self.id = Employee.new_id
    Employee.new_id += 1

class Meeting:
  def __init__(self):
    self.attendees = []
  
  def __add__(self, employee):
    print("ID {} added.".format(employee.id))
    self.attendees.append(employee)

  # Write your code
  def __len__(self):
    return len(self.attendees)
  
    
e1 = Employee()
e2 = Employee()
e3 = Employee()
m1 = Meeting()
m1 + e1 
m1 + e2 
m1 + e3 
print(len(m1))
# -> ID 1 added.
# -> ID 2 added.
# -> ID 3 added.
# -> 3

ID 1 added.
ID 2 added.
ID 3 added.
3


`m1 + e1` still calls `m1.__add__(e1)` even though the result isn't assigned — `__add__` mutates `attendees` in place. `len(m1)` is `m1.__len__()`, which just returns the length of that list. Same idea as `__add__` for `+`: a dunder hooks a built-in operation onto your class.

### OOP Pillar: Abstraction

When a program gets big, classes start sharing functionality or the inheritance structure gets hard to see. **Abstraction** helps by defining the behaviors a class structure must implement — so you don't leave things out or overlap as hierarchies grow.

In [17]:
from abc import ABC, abstractmethod

class Animal(ABC):
  def __init__(self, name):
    self.name = name

  @abstractmethod
  def make_noise(self):
    pass

class Cat(Animal):
  def make_noise(self):
    print("{} says, Meow!".format(self.name))

class Dog(Animal):
  def make_noise(self):
    print("{} says, Woof!".format(self.name))

kitty = Cat("Maisy")
doggy = Dog("Amber")
kitty.make_noise() # "Maisy says, Meow!"
doggy.make_noise() # "Amber says, Woof!"

Maisy says, Meow!
Amber says, Woof!


Two steps make `Animal` abstract (cannot be instantiated):

1. Inherit from `ABC` (Abstract Base Class).
2. Mark the empty method with `@abstractmethod`.

`Animal("Scruffy")` then raises `TypeError: Can't instantiate abstract class Animal with abstract method make_noise`.

The class still defines *what an animal is* — `__init__` requires a name; `make_noise()` exists because all animals make some noise — but it does not implement the noise, because each animal is different. Every subclass must define its own `make_noise()` or the same error fires on that subclass too.

**Exercise** — `AbstractEmployee` already has the id-counter logic; `say_id()` is marked `@abstractmethod` and left empty. `Employee` starts with no implementation.

Two errors along the way: an empty `Employee` gives `AttributeError` (no `say_id` at all). Inherit from `AbstractEmployee` but leave `pass` and you get `TypeError: Can't instantiate abstract class Employee with abstract methods say_id`. Fix: implement `say_id()`.

In [18]:
from abc import ABC, abstractmethod

class AbstractEmployee(ABC):
  new_id = 1
  def __init__(self):
    self.id = AbstractEmployee.new_id
    AbstractEmployee.new_id += 1

  @abstractmethod
  def say_id(self):
    pass

# Write your code below
class Employee(AbstractEmployee):
    def say_id(self):
      print("My id is {}".format(self.id))


e1 = Employee()
e1.say_id()
# -> My id is 1

My id is 1


Inheriting from the abstract class is not enough — the `@abstractmethod` contract is only satisfied once `Employee` actually defines `say_id()`. The id counter still lives on `AbstractEmployee` and runs through `__init__`, so `e1` gets id 1.

### OOP Pillar: Encapsulation

**Encapsulation** is the process of making methods and data hidden inside the object they relate to. Languages usually do this with **access modifiers**:

| Modifier | Syntax | In Python |
| --- | --- | --- |
| Public | `self.x` | accessible from anywhere (the default — Python has no true access control) |
| Protected | `self._x` | convention only. No error if you touch it from outside the module; the underscore tells other developers to be careful |
| Private | `self.__x` | more than convention: **name mangling** rewrites it in the background to `obj._Classname__x` |

You can still reach a mangled name from outside, but the rewrite is meant to stop inheriting classes from clashing if they define a member of the same name.

This is not the same as dunder methods. A dunder has **two leading and two trailing** underscores (`__init__`, `__add__`) and is treated differently — dunder names are **not** mangled.

**Exercise** — `dir(e)` lists every class member, including dunders. Watch where the three id flavors land in that list.

`id` is public. `_id` is the protected convention (still shows up as `_id`). `__id` is mangled to `_Employee__id`.

In [19]:
class Employee():
    def __init__(self):
        self.id = None
        # Write your code below
        self._id =  None
        self.__id =  None 



        

e = Employee()
print(dir(e))
# look at the ends: _Employee__id (mangled first), ..., _id, id

['_Employee__id', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_id', 'id']


`dir()` is a built-in that returns every member on the object. `_id` is stored under that name — the single underscore is a signal, not a rewrite. `__id` does not appear; name mangling stored it as `_Employee__id`. You can still access it as `e._Employee__id` if you insist, which is why this is hiding-from-clashes, not real privacy.

### Getters, Setters and Deleters

Getters, setters, and deleters are one way to do encapsulation in Python: the state of an attribute is handled *inside* the class, so you can check that the data is appropriate before it lands.

`_age` is module-private by convention. Three methods wrap it — get, set, and delete.

In [20]:
class Animal:
  def __init__(self, name):
    self._name = name
    self._age = None

  def get_age(self):
    return self._age

  def set_age(self, new_age):
    if isinstance(new_age, int):
      self._age = new_age
    else:
      raise TypeError

  def delete_age(self):
    del self._age
    print("_age Deleted")

a = Animal("Rufus")
print(a.get_age()) # None

a.set_age(10)
print(a.get_age()) # 10

a.delete_age() # "_age Deleted"

None
10
_age Deleted


`get_age()` just returns `self._age`. `set_age()` is the interesting one: it only assigns if `new_age` is an `int`, otherwise it raises `TypeError` — that's the encapsulation payoff, a guard at the door. `delete_age()` uses `del` to remove the attribute entirely and prints a confirmation.

Two calls from the lesson that would blow up, so they're not in the cell:

- `a.set_age("Ten")` → `TypeError` (not an int)
- `a.get_age()` after the delete → `AttributeError` (`_age` no longer exists)

**Exercise** — employees get a `_name` (defaults to `None` unless you pass a string at construction). Wrap it with a getter, setter, and deleter.

One fix from the paste: `get_name()` had `del self._name` instead of `return self._name`. That's the getter; `del_name()` is the one that deletes.

In [21]:
class Employee():
  new_id = 1
  def __init__(self, name=None):
    self.id = Employee.new_id
    Employee.new_id += 1
    self._name = name

  # Write your code below
  def get_name(self):
    return self._name

  
  def set_name(self, new_name):
    self._name = new_name

  def del_name(self):
    del self._name

e1 = Employee("Maisy")
e2 = Employee()



e1 = Employee("Maisy")
e2 = Employee()
print(e1.get_name())

e2.set_name("Fluffy")
print(e2.get_name())

e2.del_name()
# print(e2.get_name())  # AttributeError — _name is gone
# -> Maisy
# -> Fluffy

Maisy
Fluffy


`e1` gets a name at construction; `e2` starts as `None` until `set_name("Fluffy")`. After `del_name()`, `_name` is gone — a follow-up `get_name()` raises `AttributeError` (left commented so the notebook still runs).

## Q & A (captured as I go)

*Questions posed during the lesson + answers, folded in per sublesson.*

*(No questions came up on this lesson — Part 1's Q & A covers the class/dunder basics this one builds on.)*


## TL;DR

- **OOP** models real-world entities as classes with **properties** (data) and **methods** (behavior). Four pillars: **inheritance**, **polymorphism**, **abstraction**, **encapsulation**.
- **Inheritance** — `class Child(Parent)` hands the child every attribute and method of the parent. Lookup goes instance → class → up the parents until a match is found.
- **Overriding** — redefine a parent method in the child and the child's version wins. `super()` is a proxy to the parent, so `super().say_id()` *adds to* the parent's behavior inside the override instead of replacing it.
- **Chains vs. two parents** — `Manager` → `Admin` → `Employee` cascades one `super()` per level, printing in the order the calls sit in each body. With `class Hybrid(Dog, Wolf)`, `super()` resolves only to the **first** parent listed; reach the other explicitly with `Wolf.action(self)` (pass `self` yourself). Siblings can't see each other.
- **Polymorphism** — same method name, different behavior. No shared parent required: a `Robot` with `make_noise()` loops fine alongside animals. The calling code never asks what class it's holding.
- **Dunder methods** are polymorphism on operators: `a1 + a2` is `a1.__add__(a2)` (left operand's method, right passed as the argument), `len(m1)` is `m1.__len__()`, `print(obj)` is `__repr__`.
- **Abstraction** — inherit from `ABC` and mark methods `@abstractmethod`. The abstract class can't be instantiated, and a subclass stays abstract until it implements every marked method.
- **Encapsulation** — Python has no real access control. `x` is public, `_x` is a "careful, internal" convention, `__x` gets **name-mangled** to `_Classname__x` (which prevents subclass name clashes, not access). Dunders are not mangled.
- **Getters, setters, deleters** wrap a `_protected` attribute: the getter returns it, the setter can validate or type-check before assigning, the deleter runs `del self._name` — after which touching it raises `AttributeError`.
